# A basic RAG ingestion + retrieval pipeline, built from scratch

This notebook is the "after" picture from `01_naive_rag_and_token_limits.ipynb`.
Instead of stuffing an entire document into the prompt, we:

1. load a PDF and keep track of which page each bit of text came from,
2. split it into small chunks,
3. turn each chunk into an embedding vector with a local HuggingFace model,
4. store those vectors in a FAISS index,
5. embed a question and search the index for the most similar chunks,
6. hand only *those* chunks to the LLM as context.

No LangChain, no LlamaIndex - every step is plain Python so you can see
exactly what a RAG framework is doing under the hood.

**Setup:** `%pip install -q pypdf fpdf2 sentence-transformers faiss-cpu google-generativeai`


In [ ]:
# If you don't have these installed yet:
# %pip install -q pypdf fpdf2 sentence-transformers faiss-cpu google-generativeai numpy


## Cell 1 - Loader: build a small multi-page PDF, then load it page by page

We don't have a ready-made PDF handy, so we generate one: five pages, each
covering one unrelated topic in a single paragraph. Keeping exactly one
paragraph per page keeps the chunker in the next cell honest and simple.

The loader's job is basic and specific: for every page, remember **the page
number** alongside its text. That page number is metadata we carry through
the entire pipeline so that, at the end, we can tell the user which page an
answer came from.


In [ ]:
import os
from fpdf import FPDF
from pypdf import PdfReader

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
PDF_PATH = os.path.join(DATA_DIR, "mini_textbook.pdf")

PAGES_TEXT = [
    "The water cycle describes the continuous movement of water between the oceans, the atmosphere, and the land. The sun drives evaporation from oceans and lakes, water vapor cools and condenses into clouds, and it eventually falls back down as precipitation before collecting in rivers and groundwater and flowing back to the sea.",
    "Photosynthesis is the process plants use to turn sunlight, water, and carbon dioxide into glucose and oxygen. It happens mainly inside chloroplasts, using a green pigment called chlorophyll to capture light energy, and it is the foundation of almost every food chain on Earth.",
    "The Solar System consists of the Sun and everything bound to it by gravity, including eight planets, their moons, and countless asteroids and comets. The four inner planets are small and rocky, while the four outer planets are much larger and made mostly of gas and ice.",
    "The human digestive system breaks food down into nutrients the body can absorb. Digestion starts in the mouth with chewing and saliva, continues in the stomach where acid and enzymes break food down further, and finishes in the small and large intestines, where nutrients and water are absorbed.",
    "Plate tectonics explains how Earth's outer shell is broken into large plates that slowly move over the layer beneath them. Where plates collide, push apart, or slide past each other, they cause earthquakes, volcanic activity, and the formation of mountain ranges over millions of years.",
]

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
for text in PAGES_TEXT:
    pdf.add_page()
    pdf.set_font("Helvetica", size=12)
    pdf.multi_cell(0, 8, text)
pdf.output(PDF_PATH)

print(f"Generated a {len(PAGES_TEXT)}-page PDF at {PDF_PATH}")


In [ ]:
reader = PdfReader(PDF_PATH)

# Very basic loader: one entry per page, keeping the page number as metadata.
pages = [
    {"page": page_number + 1, "text": page.extract_text()}
    for page_number, page in enumerate(reader.pages)
]

for p in pages:
    print(f"--- page {p['page']} ---")
    print(p["text"])
    print()


## Cell 2 - Chunker: split each page's text on blank lines ("\n\n") only

This is the simplest possible chunker: no token counting, no overlap, no
sentence splitting - just "wherever there's a blank line, that's a new
chunk." Real PDF text extraction often collapses blank lines, which is why
our generated PDF above uses exactly one paragraph per page: it keeps this
basic splitter meaningful without extra cleanup logic. Each chunk keeps the
page number it came from.


In [ ]:
def chunk_page(page_text: str, page_number: int, next_id: int):
    paragraphs = [p.strip() for p in page_text.split("\n\n") if p.strip()]
    return [
        {"chunk_id": next_id + i, "page": page_number, "text": paragraph}
        for i, paragraph in enumerate(paragraphs)
    ]

chunks = []
for page in pages:
    chunks.extend(chunk_page(page["text"], page["page"], len(chunks)))

print(f"Total chunks: {len(chunks)}\n")
for c in chunks:
    print(c)


## Cell 3 - Embeddings: load a HuggingFace sentence-embedding model

We use `sentence-transformers`, a thin wrapper around HuggingFace models that
turns text into fixed-length numeric vectors. Before we index anything, we
need to know the vector size (dimension) the model produces, since the
vector database needs that number up front.


In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

dimension = embed_model.get_embedding_dimension()
print(f"Loaded '{EMBED_MODEL_NAME}'")
print(f"Embedding dimension: {dimension}")


## Cell 4 - Vector DB: create a FAISS index

FAISS (Facebook AI Similarity Search) stores vectors and can quickly find the
ones most similar to a query vector. `IndexFlatIP` compares vectors with a
plain dot product; since we'll L2-normalize every embedding we add, that dot
product behaves exactly like cosine similarity - a simple, exact index with
no approximation, which is all we need at this scale. (ChromaDB is a common
alternative that does the same job with a slightly higher-level API.)


In [ ]:
import faiss

index = faiss.IndexFlatIP(dimension)
print(index)
print(f"Index ready: {index.is_trained}, vectors currently stored: {index.ntotal}")


## Cell 5 - Embed every chunk, and store it in two matching pieces

A vector index only ever sees numbers - it has no idea what those numbers
mean. So we build two aligned structures:

- `embeddings`: a NumPy array of shape `(num_chunks, dimension)` - one vector per chunk.
- `metadata`: a dictionary mapping each vector's position back to the real text and page number.

Position `i` in `embeddings` always corresponds to key `i` in `metadata`.


In [ ]:
import numpy as np

chunk_texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(chunk_texts, normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype="float32")

metadata = {
    i: {"chunk_id": c["chunk_id"], "page": c["page"], "text": c["text"]}
    for i, c in enumerate(chunks)
}

print(f"Embeddings array shape: {embeddings.shape}")
print(f"Metadata entries: {len(metadata)}")


## Cell 6 - Add the vectors to the index


In [ ]:
index.add(embeddings)
print(f"Vectors now stored in the index: {index.ntotal}")


## Cell 7 - Query: embed the question, score it against every chunk

We embed the question with the *same* model used for the chunks - query and
chunks have to live in the same vector space for similarity to mean
anything. `index.search(..., k=index.ntotal)` asks FAISS to return a
similarity score for every single stored vector, not just the best one, so
we can see the full ranking before deciding how many to keep.


In [ ]:
query = "How does photosynthesis work?"

query_vector = embed_model.encode([query], normalize_embeddings=True).astype("float32")
scores, indices = index.search(query_vector, k=index.ntotal)

print(f"Similarity score for every chunk, for the query: {query!r}\n")
for score, idx in zip(scores[0], indices[0]):
    print(f"  chunk {idx} (page {metadata[idx]['page']}): {score:.4f}")


## Cell 8 - Keep only the top 3 chunks

`scores` and `indices` from `index.search` already come back sorted, best
match first, so the top 3 are just the first 3 entries. We use the indices to
look the real text back up in `metadata`.


In [ ]:
TOP_K = 3
top_indices = indices[0][:TOP_K]
top_scores = scores[0][:TOP_K]
top_chunks = [metadata[idx] for idx in top_indices]

print(f"Top {TOP_K} chunks for {query!r}:\n")
for score, chunk in zip(top_scores, top_chunks):
    print(f"[page {chunk['page']}, score {score:.4f}] {chunk['text']}\n")


## Cell 9 - Club the retrieved chunks into a single context, and build the prompt

Same f-string pattern as the naive notebook - the only difference is that the
reference text is now three short, relevant chunks instead of an entire
document.


In [ ]:
context = "\n\n".join(f"(Page {c['page']}) {c['text']}" for c in top_chunks)

prompt = f"""You are a helpful assistant.
Answer the question using ONLY the reference text below. If the answer is not
contained in the reference, say you don't know. Mention which page(s) you used.

Reference:
\"\"\"
{context}
\"\"\"

Question: {query}
"""

print(prompt)


## Cell 10 - Send the prompt to the LLM

Set `GOOGLE_API_KEY` as an environment variable (or paste it below) using a
key from https://aistudio.google.com/apikey.


In [ ]:
import os
import google.generativeai as genai

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "PASTE_YOUR_API_KEY_HERE")
genai.configure(api_key=GOOGLE_API_KEY)

llm = genai.GenerativeModel("gemini-1.5-flash")
response = llm.generate_content(prompt)
print(response.text)


## Compare with notebook 1

In `01_naive_rag_and_token_limits.ipynb`, answering a question meant sending
the *entire* document (or worse, an entire book) on every request. Here, we
sent 3 short chunks - a handful of sentences - and got an answer grounded in
exactly the right page. That's the whole value proposition of RAG: retrieval
lets the LLM's context window stay small, fast, and relevant, no matter how
large the underlying document collection grows.
